# API Functions → Binary ML Dataset

Flattens `API Functions.csv` into a binary matrix suitable for machine learning:
- **Rows**: one per SHA256 sample
- **API columns**: 1 if that API function is present in the sample, 0 otherwise
- **Label columns**: one-hot encoded malware type (Backdoor, Downloader, Generic Malware, Ransomware, Spyware, …)

**Before running:** upload `API Functions.csv` to the Google Drive folder specified in the *Configuration* cell.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Imports

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

print(f"pandas  {pd.__version__}")

## Step 3 — Configuration

Edit `DRIVE_BASE` to match the folder inside your Google Drive where you placed `API Functions.csv`.

In [ ]:
# ── EDIT THIS PATH to match your Google Drive folder ──────────────────────────
DRIVE_BASE  = "/content/drive/MyDrive/ZIET8025/dataset"
# ──────────────────────────────────────────────────────────────────────────────

INPUT_FILE  = os.path.join(DRIVE_BASE, "API Functions.csv")
OUTPUT_FILE = os.path.join(DRIVE_BASE, "API_Functions_flat.csv")

assert os.path.exists(INPUT_FILE), (
    f"File not found: {INPUT_FILE}\n"
    "Please check DRIVE_BASE and make sure 'API Functions.csv' is in that folder."
)
print(f"Input  : {INPUT_FILE}")
print(f"Output : {OUTPUT_FILE}")

## Step 4 — Load CSV

In [ ]:
print("Reading CSV (this may take 30–60 s for ~18 k rows) …")

df = pd.read_csv(INPUT_FILE, header=None, index_col=0, low_memory=False)
df.index.name = "sha256"

# Col 0 is already the index (SHA256).
# Col 1 (iloc 0 after index_col) → malware type label.
# Cols 2+ (iloc 1+) → API function name columns, padded with empty strings.
label_col = df.iloc[:, 0]
api_cols  = df.iloc[:, 1:]

print(f"Rows loaded : {len(df):,}")
print(f"Label sample: {label_col.value_counts().to_dict()}")
print(f"API columns : {api_cols.shape[1]:,} (including padding)")

## Step 5 — Build Binary API Presence Matrix

Each row becomes a **set** of API names (empty strings and literal `"NA"` entries are excluded). `MultiLabelBinarizer` then converts these sets into a 0/1 matrix with one column per unique API name.

In [ ]:
EXCLUDED = {"NA", ""}

def row_to_api_set(row):
    """Return the set of real API names present in this row."""
    return {v for v in row.dropna().astype(str) if v not in EXCLUDED}

print("Parsing API sets per row …")
api_sets = api_cols.apply(row_to_api_set, axis=1)

print("Building binary API presence matrix …")
mlb = MultiLabelBinarizer(sparse_output=False)
api_matrix = pd.DataFrame(
    mlb.fit_transform(api_sets),
    index=df.index,
    columns=mlb.classes_,
    dtype="int8",
)

print(f"API matrix shape : {api_matrix.shape}  "
      f"({api_matrix.shape[0]:,} samples × {api_matrix.shape[1]:,} unique APIs)")
print(f"Memory usage     : {api_matrix.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## Step 6 — One-Hot Encode Malware Labels

In [ ]:
label_matrix = pd.get_dummies(label_col.rename("label"), dtype="int8")
label_matrix.index = df.index

print(f"Label columns : {list(label_matrix.columns)}")
print(label_matrix.sum().rename("sample_count").to_string())

## Step 7 — Concatenate and Save

The final CSV has:
- `sha256` as the row index
- All unique API function name columns (0/1)
- All malware type columns (0/1) at the right-hand side

In [ ]:
result = pd.concat([api_matrix, label_matrix], axis=1)

print(f"Final dataset shape : {result.shape}  "
      f"({result.shape[0]:,} rows × {result.shape[1]:,} columns)")
print(f"Total memory        : {result.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\nSaving to {OUTPUT_FILE} …")
result.to_csv(OUTPUT_FILE)
print("Done.")

## (Optional) Preview the Result

In [ ]:
result.head(3)